## Rough Bergomi Volatility
Train a neural network to replicate the mapping from Rough Bergomi parameters to implied volatility

In [ ]:
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import grad
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
from joblib import dump
from scipy.optimize import minimize
import math

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load Data

In [ ]:
base_filename = 'bergomi_option_grid'

In [ ]:
bergomi_data_list = list()
for i in range(0,20):
    filename = base_filename + str(i) + '.csv'
    if os.path.exists(filename):
        new_bergomi_data = pd.read_csv(filename, index_col=[0])
        bergomi_data_list.append(new_bergomi_data) 
    
bergomi_data = pd.concat(bergomi_data_list)
bergomi_data.reset_index(inplace=True)

In [ ]:
bergomi_data.head()

In [ ]:
y = bergomi_data['iv'].to_numpy()

In [ ]:
X = bergomi_data.drop(['index', 'iv', 'time'], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.02, random_state=42)

In [ ]:
standard_scaler = StandardScaler()
X_train = standard_scaler.fit_transform(X_train)
X_test = standard_scaler.transform(X_test)

In [ ]:
dump(standard_scaler, 'rbergomi_scaler_grid.bin', compress=True)

In [ ]:
batch_size = 1024

In [ ]:
train_x = torch.Tensor(X_train).to(device)
train_y = torch.Tensor(y_train).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

test_x = torch.Tensor(X_test).to(device)
test_y = torch.Tensor(y_test).to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

### Train Model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    grad_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        for batch_X, batch_y in train_loader:

            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)

        # Evaluation on the test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.6f}, Test Loss: {test_loss:.6f}")
    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super(Swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, width = 32, use_batch_norm=False):
        super(NeuralNetwork, self).__init__()
        self.use_batch_norm = use_batch_norm
        self.activation = Swish()
        self.fc1 = nn.Linear(6, width)
        self.bn1 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc2 = nn.Linear(width, width)
        self.bn2 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc3 = nn.Linear(width, width)
        self.bn3 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc4 = nn.Linear(width, width)
        self.bn4 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc5 = nn.Linear(width, 1)

    def forward(self, x):
        x = self.activation(self.bn1(self.fc1(x)))
        x = self.activation(self.bn2(self.fc2(x)))
        x = self.activation(self.bn3(self.fc3(x)))
        x = self.activation(self.bn4(self.fc4(x)))
        x = self.fc5(x)
        return x

In [ ]:
no_epochs = 1000
loss_fn = nn.MSELoss()

In [ ]:
bergominn = NeuralNetwork(256, True).to(device)
optimizer = optim.Adam(bergominn.parameters(), lr=0.001)

In [ ]:
history = train_model(bergominn, train_loader, test_loader, 
                          loss_fn, optimizer, no_epochs)

In [ ]:
model_path = "bergominn_grid2.pth"

In [ ]:
torch.save(bergominn.state_dict(), model_path)

### Plot Losses

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

train_loss_values = history['train_loss']
test_loss_values = history['test_loss']
ax.plot(train_loss_values, color='black', linestyle='-',label = 'training loss', )
ax.plot(test_loss_values, color='black', linestyle='--', label = 'test loss')

ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.set_ylim(0,0.0005)
ax.legend()
fig.savefig('BergomiNNLossGrid2.png', dpi=300, bbox_inches='tight')

### Plot Sample Smiles

In [ ]:
mats = np.round(np.linspace(0.1,2,8),1)
mats

In [ ]:
strikes = np.round(np.linspace(0.7,1.3,7),1)
strikes

In [ ]:
from py_vollib.black_scholes.implied_volatility import implied_volatility
import rbergomi

def rBergomiIV(S0, K, T, rho, eta, H, xi_0, num_paths, num_steps):
    a = H - 0.5
    rB = rbergomi.rBergomi(n = num_steps, N = num_paths, T = T, a = a)
    dW1 = rB.dW1()
    dW2 = rB.dW2()
    Y = rB.Y(dW1)
    dB = rB.dB(dW1, dW2, rho = rho)
    V = rB.V(Y, xi = xi_0, eta = eta)
    S = rB.S(V, dB)
    ST = S[:,-1][:,np.newaxis]
    call_payoff = np.maximum(ST - K,0)
    call_price = np.mean(call_payoff, axis = 0)[:,np.newaxis].item()
    iv = implied_volatility(call_price, S0, K, T, 0.0, 'c')
    return iv

In [ ]:
S0 = 1
eta = 2.5
rho = -0.5
H = 0.25
xi_0 = 0.08

In [ ]:
bergomi_smile_lst = list()

In [ ]:
for imat in mats:
    smile_lst = list()
    for ik in strikes:
        iv = rBergomiIV(S0, ik, imat, rho, eta, H, xi_0, 30000, 100)
        smile_lst.append(iv)
    bergomi_smile_lst.append(smile_lst)

In [ ]:
bergomi_smile = np.array(bergomi_smile_lst)
bergomi_smile

In [ ]:
bergominn_smile_lst = list()

In [ ]:
for imat in mats:
    for ik in strikes:
        input_lst = list()
        input_lst.append(eta)
        input_lst.append(rho)
        input_lst.append(H)
        input_lst.append(xi_0)
        input_lst.append(imat)
        input_lst.append(ik)
        bergominn_smile_lst.append(input_lst)

In [ ]:
heading_list = ['eta', 'rho', 'H', 'xi_0', 'tau', 'K']
df = pd.DataFrame(bergominn_smile_lst, columns=heading_list)
df.head()

In [ ]:
bergomi_X = standard_scaler.transform(df)

In [ ]:
tbergomi_X = torch.Tensor(bergomi_X).to(device)

In [ ]:
bergominn_iv = bergominn(tbergomi_X)

In [ ]:
bergominn_iv.shape

In [ ]:
bergominn_iv = bergominn_iv.reshape((8, 7)).detach().cpu().numpy()

In [ ]:
bergominn_iv

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(14, 10))
axs = axs.flatten()

for i, maturity in enumerate(mats):
    axs[i].plot(strikes, bergomi_smile[i], color='black', linestyle = '-', label='rBergomi')
    axs[i].plot(strikes, bergominn_iv[i], color='black', linestyle = '--', label='DNN')
    axs[i].set_title(f'Maturity = {maturity} years')
    axs[i].set_xlabel('Relative Strike (K/S)')
    axs[i].set_ylabel('Implied Volatility')
    axs[i].legend()

fig.savefig('BergomiSmilesGrid2.png', dpi=300, bbox_inches='tight')

### Load and prepare the vol surface

In [ ]:
in_file = 'SPXOptions.csv'
dfopt = pd.read_csv(in_file)

In [ ]:
dfvol = dfopt[dfopt['quote_date'] == '2013-03-27']
dfvol = dfvol[dfvol['type'] == 'put']
underlying = dfvol['underlying'].unique()

In [ ]:
drop_list = ['contract', 'underlying', 'expiration', 'type', 'style',
       'bid', 'ask', 'volume', 'open_interest', 'quote_date', 'delta', 'gamma',
       'theta', 'vega', 'rate', '20D_HV','40D_HV', '60D_HV', 'BlackScholes', 'mid', 'error']
dfvol.drop(drop_list, axis=1, inplace=True)

In [ ]:
dfvol['relative_strike'] = dfvol['strike'] / underlying.item()

In [ ]:
dfvol.drop(['strike'], axis=1, inplace=True)

In [ ]:
dfvol = dfvol[dfvol['relative_strike'] < 1.5]
dfvol = dfvol[dfvol['relative_strike'] > 0.5]
dfvol = dfvol[dfvol['maturity'] > 0.4]
dfvol = dfvol[dfvol['maturity'] <= 2.0]

In [ ]:
dfvol['maturity'].unique()

### Fit to the Surface

In [ ]:
class rbergomiiv:
    def __init__(self, model, sc, device):
        self.model = model
        self.sc = sc
        self.device = device
    
    def impliedvol(self, eta, rho, H, xi_0, mats, strikes):
        no_opt = len(mats)
        veta = np.full(no_opt, eta)
        vrho = np.full(no_opt, rho)
        vH = np.full(no_opt, H)
        vxi_0 = np.full(no_opt, xi_0)
        param_dict = {'eta': veta, 'rho': vrho,
                      'H': vH, 'xi_0': vxi_0,
                      'tau': mats, 'K': strikes}       
        df = pd.DataFrame(param_dict)
        bergomi_X = self.sc.transform(df)
        tbergomi_X = torch.Tensor(bergomi_X).to(self.device)
        bergominn_iv = self.model(tbergomi_X)
        return bergominn_iv.squeeze().detach().cpu().numpy()  

In [ ]:
def objective_function(params, iv_surface, rbergomiobj):
    total_error = 0
    eta = params[0]
    rho = params[1]
    H = params[2]
    xi_0 = params[3]
    mats = iv_surface['maturity']
    strikes = iv_surface['relative_strike']
    model_iv = rbergomiobj.impliedvol(eta, rho, H, xi_0, mats, strikes)
    market_iv = iv_surface['implied_volatility'].to_numpy()
    mse = np.mean((market_iv - model_iv) ** 2)
    return mse

In [ ]:
param_lst = [eta, rho, H, xi_0]
initial_params = np.array(param_lst)

In [ ]:
bergomiivobj = rbergomiiv(bergominn, standard_scaler, device)

In [ ]:
tmats = [1.0, 1.0]
tstrikes = [1.0, 1.0]

In [ ]:
bergomiivobj.impliedvol(eta, rho, H, xi_0, tmats, tstrikes)

In [ ]:
optimization_method = 'Nelder-Mead'

In [ ]:
bounds = [[0.5, 4.0], [-0.95, -0.1], [0.025, 0.5], [0.01, 0.16]]

In [ ]:
result = minimize(objective_function, initial_params, args=(dfvol,bergomiivobj), 
                  bounds=bounds, method=optimization_method, tol=1e-10)

In [ ]:
if result.success:
    fitted_params = result.x
    print("Optimization succeeded.")
    print(f"Fitted rBergomi parameters: {fitted_params}")
else:
    print("Optimization failed.")
    print(result.message)

In [ ]:
feta, frho, fH, fxi_0 = result.x

In [ ]:
rbiv = bergomiivobj.impliedvol(feta, frho, fH, fxi_0, dfvol['maturity'], dfvol['relative_strike'])

In [ ]:
model_ivd = {'implied_volatility': rbiv, 'maturity': dfvol['maturity'].to_numpy(), 
             'relative_strike': dfvol['relative_strike'].to_numpy()}

In [ ]:
model_dfvol = pd.DataFrame(model_ivd)

In [ ]:
from py_vollib.black_scholes.implied_volatility import implied_volatility
import rbergomi

def rBergomiIV(S0, K, T, rho, eta, H, xi_0, num_paths, num_steps):
    a = H - 0.5
    rB = rbergomi.rBergomi(n = num_steps, N = num_paths, T = T, a = a)
    dW1 = rB.dW1()
    dW2 = rB.dW2()
    Y = rB.Y(dW1)
    dB = rB.dB(dW1, dW2, rho = rho)
    V = rB.V(Y, xi = xi_0, eta = eta)
    S = rB.S(V, dB)
    ST = S[:,-1][:,np.newaxis]
    call_payoff = np.maximum(ST - K,0)
    call_price = np.mean(call_payoff, axis = 0)[:,np.newaxis].item()
    iv = implied_volatility(call_price, S0, K, T, 0.0, 'c')
    return iv

In [ ]:
dfvolmats = dfvol['maturity'].unique()
dfvolstrikes = dfvol['relative_strike'].unique()

In [ ]:
dfvolstrikes

In [ ]:
dfrbermgomiform = pd.DataFrame()

In [ ]:
bergomi_smile_lst = list()

for imat in dfvolmats:
    smile_lst = list()
    for ik in strikes:
        try:
            iv = rBergomiIV(S0, ik, imat, frho, feta, fH, fxi_0, 30000, 100)
            smile_lst.append(iv)
        except:
            continue
    bergomi_smile_lst.append(smile_lst)

bergomi_smile = np.array(bergomi_smile_lst)
bergomi_smile

In [ ]:
len(bergomi_smile)

In [ ]:
unique_maturities = pd.concat([dfvol['maturity'], model_dfvol['maturity']]).unique()
unique_maturities.sort()  # Sort maturities for a more logical plot arrangement

# Determine the grid size for subplots
n = len(unique_maturities)
cols = 4  # You can adjust this value based on your preference
rows = math.ceil(n / cols)

# Step 2: Create subplots for each maturity
plt.figure(figsize=(14, 10))
for i, maturity in enumerate(unique_maturities, start=1):
    plt.subplot(rows, cols, i)
    
    # Filter data for the current maturity
    market_subset = dfvol[dfvol['maturity'] == maturity]
    model_subset = model_dfvol[model_dfvol['maturity'] == maturity]
    #form_subset = dfrbermgomiform[dfrbermgomiform['maturity'] == maturity]

    # Make sure data is sorted by relative_strike for smooth lines
    market_subset = market_subset.sort_values('relative_strike')
    model_subset = model_subset.sort_values('relative_strike')
    #form_subset = form_subset.sort_values('relative_strike')

    # Plotting
    plt.plot(market_subset['relative_strike'], market_subset['implied_volatility'], color='black', label='Market', linestyle='-')
    plt.plot(model_subset['relative_strike'], model_subset['implied_volatility'], color='black', label='DNN', linestyle='--')
    plt.plot(strikes, bergomi_smile[i-1], color='black', label='MC', linestyle='-.')

    plt.title(f'Maturity = {maturity} years')
    plt.xlabel('Relative Strike (K/S)')
    plt.ylabel('Implied Volatility')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.savefig('SPXBergomi3.png', dpi=300, bbox_inches='tight')